# Get Started with the Foxglove Notebook Integration

[Foxglove](https://foxglove.dev/) is a multimodal robotics observability platform. The [Python SDK](https://docs.foxglove.dev/docs/getting-started/python) ships an optional **notebook integration** that lets you embed a fully-featured Foxglove viewer directly inside Jupyter, JupyterLab, Google Colab, or VS Code.

This notebook is a runnable companion to the [Foxglove notebook integration docs](https://docs.foxglove.dev/docs/notebook). It walks through:

1. Installing the integration
2. Creating a notebook buffer and logging messages
3. Building layouts programmatically with the `foxglove.layouts` API


## Install the integration

The notebook integration is shipped as the optional `notebook` extra of the [`foxglove-sdk`](https://pypi.org/project/foxglove-sdk/) package. In Colab or any other fresh kernel, install it like this:

In [ ]:
%pip install -q "foxglove-sdk[notebook]"

## Using the integration

Once the integration is installed, create a notebook buffer with [`foxglove.init_notebook_buffer()`](https://docs.foxglove.dev/docs/notebook#using-the-integration). The buffer collects every message you log to the SDK's default context. When you're ready to visualize the data, call `nb_buffer.show()` &mdash; **it must be the last expression in the cell** so Jupyter renders the embedded widget.

> ⚠️ **Always set `log_time` explicitly** when calling `foxglove.log(...)`. If you don't, the SDK uses the wall-clock time at the moment each message is logged, which can lead to surprising playback behavior in the viewer.

The example below logs 300 `count` messages on the `/hello` topic, spaced 33&nbsp;ms apart (roughly 10 seconds of buffered data), then displays the embedded Foxglove viewer.

In [ ]:
import foxglove
from foxglove.layouts import Layout, RawMessagesConfig, RawMessagesPanel

# Create a notebook buffer to collect messages.
nb_buffer = foxglove.init_notebook_buffer()

# Log 300 messages spaced 33ms apart (~10 seconds of data).
for t in range(10 * 30):
    timestamp = t * 0.033
    foxglove.log("/hello", {"count": t}, log_time=int(timestamp * 1e9))

# Pre-configure a Raw Messages panel pointed at /hello so the embedded
# viewer shows the buffered data immediately instead of an empty default layout.
layout = Layout(
    content=RawMessagesPanel(
        title="/hello",
        config=RawMessagesConfig(topic_path="/hello"),
    ),
)

# Display the data in the embedded Foxglove viewer.
# This must be the last expression in the cell for the widget to render.
nb_buffer.show(layout=layout)

### Logging richer data

The SDK ships typed channels for the [Foxglove schemas](https://docs.foxglove.dev/docs/visualization/message-schemas/introduction) so you can log 3D scenes, images, point clouds, and more &mdash; and they all render in the embedded viewer just like they would in the desktop app.

Here we log an animated cube on the `/scene` topic. We'll reuse this data later when we build a custom layout.

In [ ]:
import math

import foxglove
from foxglove.channels import SceneUpdateChannel
from foxglove.messages import (
    Color,
    CubePrimitive,
    SceneEntity,
    SceneUpdate,
    Vector3,
)

# Start with a fresh buffer so this cell stands on its own.
nb_buffer = foxglove.init_notebook_buffer()

scene_channel = SceneUpdateChannel("/scene")

# Animate a cube whose size pulses over ~10 seconds.
for t in range(10 * 30):
    timestamp = t * 0.033
    size = abs(math.sin(timestamp)) + 1.0

    # Also log a scalar so we have something to plot later.
    foxglove.log("/hello", {"count": t}, log_time=int(timestamp * 1e9))

    scene_channel.log(
        SceneUpdate(
            entities=[
                SceneEntity(
                    cubes=[
                        CubePrimitive(
                            size=Vector3(x=size, y=size, z=size),
                            color=Color(r=1.0, g=0.8, b=0.0, a=1.0),
                        ),
                    ],
                ),
            ],
        ),
        log_time=int(timestamp * 1e9),
    )

nb_buffer.show()

## Layout management

A [layout](https://docs.foxglove.dev/docs/visualization/layouts/) is the arrangement of panels &mdash; 3D scenes, plots, images, raw messages, etc. &mdash; that turns raw data into an opinionated view of your robot. The SDK ships a Python API in [`foxglove.layouts`](https://docs.foxglove.dev/docs/notebook/layouts) for building layouts programmatically and passing them to the embedded viewer via `nb_buffer.show(layout=...)`.

The simplest layout is a single panel. Below we build one with a `MarkdownPanel` plus a global variable, mirroring the example from the docs.

In [ ]:
import foxglove
from foxglove.layouts import Layout, MarkdownConfig, MarkdownPanel

layout = Layout(
    content=MarkdownPanel(
        config=MarkdownConfig(markdown="Hello, world!"),
    ),
    variables={"my_variable": 1},
)

nb_buffer = foxglove.init_notebook_buffer()
nb_buffer.show(layout=layout)

### Composing panels with containers

Real layouts usually have multiple panels arranged side-by-side or in tabs. The `foxglove.layouts` module ships three containers for that:

- [`SplitContainer`](https://docs.foxglove.dev/docs/notebook/layouts) &mdash; arranges   children side-by-side in a `"row"` or `"column"`.
- [`TabContainer`](https://docs.foxglove.dev/docs/notebook/layouts) &mdash; arranges   children in tabs, with one visible at a time.
- [`StackContainer`](https://docs.foxglove.dev/docs/notebook/layouts) &mdash; arranges   children in a vertically-scrolling list.

Below we re-log the cube and counter from the previous cells, then build a layout that puts the 3D scene and a plot of `/hello.count` side-by-side inside a tab. This gives us a small but realistic example of nested containers driving real data.

In [ ]:
import math

import foxglove
from foxglove.channels import SceneUpdateChannel
from foxglove.layouts import (
    BaseRendererGridLayerSettings,
    BaseRendererSceneUpdateTopicSettings,
    Layout,
    PlotConfig,
    PlotPanel,
    PlotSeries,
    SplitContainer,
    SplitItem,
    TabContainer,
    TabItem,
    ThreeDeeConfig,
    ThreeDeePanel,
)
from foxglove.messages import (
    Color,
    CubePrimitive,
    SceneEntity,
    SceneUpdate,
    Vector3,
)

# Fresh buffer + re-log the data so this cell is self-contained.
nb_buffer = foxglove.init_notebook_buffer()
scene_channel = SceneUpdateChannel("/scene")

for t in range(10 * 30):
    timestamp = t * 0.033
    size = abs(math.sin(timestamp)) + 1.0

    foxglove.log("/hello", {"count": t}, log_time=int(timestamp * 1e9))

    scene_channel.log(
        SceneUpdate(
            entities=[
                SceneEntity(
                    cubes=[
                        CubePrimitive(
                            size=Vector3(x=size, y=size, z=size),
                            color=Color(r=0.1, g=0.7, b=1.0, a=1.0),
                        ),
                    ],
                ),
            ],
        ),
        log_time=int(timestamp * 1e9),
    )

# A 3D panel showing the cube on /scene, with a base grid layer for context.
scene_panel = ThreeDeePanel(
    title="Scene",
    config=ThreeDeeConfig(
        layers={"grid": BaseRendererGridLayerSettings()},
        topics={"/scene": BaseRendererSceneUpdateTopicSettings(visible=True)},
    ),
)

# A plot panel showing /hello.count over time.
plot_panel = PlotPanel(
    title="Counter",
    config=PlotConfig(
        paths=[PlotSeries(value="/hello.count", enabled=True)],
        show_legend=True,
    ),
)

# Side-by-side row, wrapped in a single-tab TabContainer to demonstrate both APIs.
split = SplitContainer(
    direction="row",
    items=[
        SplitItem(content=scene_panel),
        SplitItem(content=plot_panel),
    ],
)
tabs = TabContainer(tabs=[TabItem(title="Overview", content=split)])

layout = Layout(content=tabs)
nb_buffer.show(layout=layout)

## Where to next

- 📖 [Notebook integration docs](https://docs.foxglove.dev/docs/notebook) &mdash; the   full reference, including buffer management and updating an existing viewer with   `viewer.refresh()`.
- 🧱 [Layouts API reference](https://docs.foxglove.dev/docs/notebook/layouts) &mdash;   every panel and container the `foxglove.layouts` module ships.
- 🐍 [Python SDK getting started](https://docs.foxglove.dev/docs/getting-started/python)   &mdash; using the SDK outside of notebooks.
- 📊 [`data_platform` notebook](../data_platform/README.md) &mdash; a more advanced   example that pulls data from Foxglove Data Management for offline analysis.
- 💬 [Foxglove Discord](https://foxglove.dev/community) &mdash; questions, feedback, and   what we're working on next.
